# Ensemble Methods & Boosting — AI Lab Instructor Notebook

*Classical ML · Medium*

Understand why ensembles work (variance reduction, bias reduction), and learn interview-grade intuition for Random Forests, Gradient Boosting, and XGBoost-style thinking.

**Outcome.** Students can:
- explain bagging vs boosting,
- train and tune sklearn RandomForest and GradientBoosting,
- compare against a baseline model,
- reason about overfitting, leakage, and feature importance.

6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Ensemble Methods & Boosting — Student Lab

Week 4 introduces sklearn models, but you must still explain *why* they work (bias/variance).

In [ ]:
$$f

**Why this works.** # Ensemble Methods & Boosting — Instructor (Solutions + Rationale)

**OOB Score (Out-Of-Bag Score)**

OOB score is an internal validation score used in bagging-based models like Random Forest.

It estimates test performance without needing a separate validation set.

**Why “Out-Of-Bag” (oob_score)?**

In bagging:

Each tree is trained on a bootstrap sample (sample with replacement).

On average, about 63% of training samples are used to train a tree.

The remaining ~37% are not seen by that tree.

Those unused samples are called:

Out-Of-Bag (OOB) samples

## Task 2: Choose dataset
*Section: Dataset (synthetic default, real optional)*

## Section 0 — Dataset (synthetic default, real optional)

![Dataset Choice Diagram](/images/w4-d3-dataset-choice.svg)

### Task 0.1: Choose dataset
Use synthetic by default. Optionally switch to breast cancer dataset.

In [ ]:
use_real = False

if use_real:
    data = load_breast_cancer()
    X = data.data
    y = data.target
else:
    X, y = make_classification(
        n_samples=2000,
        n_features=20,
        n_informative=8,
        n_redundant=4,
        class_sep=1.0,
        flip_y=0.03,
        random_state=0,
    )

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

## Task 3: Train baseline decision tree vs random forest
*Section: Baseline vs Trees vs Random Forest*

## Section 1 — Baseline vs Trees vs Random Forest

### Task 1.1: Train baseline decision tree vs random forest

- DecisionTreeClassifier(max_depth=?)
- RandomForestClassifier(n_estimators=?, max_depth=?, oob_score=True, bootstrap=True)

Compute accuracy + ROC-AUC on validation.

**Checkpoint:** Why does bagging reduce variance?

In [ ]:
def eval_model(clf, X, y):
    pred = clf.predict(X)
    acc = accuracy_score(y, pred)
    if hasattr(clf, 'predict_proba'):
        proba = clf.predict_proba(X)[:, 1]
        auc = roc_auc_score(y, proba)
    else:
        auc = float('nan')
    return float(acc), float(auc)

tree = DecisionTreeClassifier(max_depth=3, random_state=0)
rf = RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_leaf=2, oob_score=True, bootstrap=True, random_state=0, n_jobs=-1)

tree.fit(Xtr, ytr)
rf.fit(Xtr, ytr)
print('tree', eval_model(tree, Xva, yva))
print('rf  ', eval_model(rf, Xva, yva))
print('rf oob_score', rf.oob_score_)

# Rationale: bagging averages noisy high-variance trees -> lower variance and better generalization.

**Why this works.** bagging averages noisy high-variance trees -> lower variance and better generalization.

## Task 4: Feature importance gotcha
*Section: Baseline vs Trees vs Random Forest*

### Task 1.2: Feature importance gotcha

Inspect `feature_importances_` and explain why correlated features can distort importances.

In [ ]:
imp = rf.feature_importances_
top = np.argsort(-imp)[:10]
print('top idx', top)
print('top importances', imp[top])

# Rationale: impurity-based importance can split credit across correlated features; use permutation importance for robustness.

**Why this works.** impurity-based importance can split credit across correlated features; use permutation importance for robustness.

## Task 5: Train GradientBoostingClassifier
*Section: Gradient Boosting*

## Section 2 — Gradient Boosting

### Task 2.1: Train GradientBoostingClassifier


**Checkpoint:** Why can boosting overfit with too many estimators?

In [ ]:
settings = [
    {'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 2},
    {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 2},
    {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 2},
]

for s in settings:
    gb = GradientBoostingClassifier(random_state=0, **s)
    gb.fit(Xtr, ytr)
    print('gb', s, eval_model(gb, Xva, yva))

# Rationale: boosting fits residuals; learning rate shrinks each step; too many estimators can overfit.

**Why this works.** boosting fits residuals; learning rate shrinks each step; too many estimators can overfit.

## Task 6: Explain what each knob does
*Section: XGBoost-style knobs (conceptual)*

## Section 3 — XGBoost-style knobs (conceptual)

### Task 3.1: Explain what each knob does
Write 2-3 bullets each:
- subsample
- colsample
- learning rate
- max_depth

In [ ]:
# subsample
# - Fraction of training rows used to build each tree (row sampling).
# - Adds randomness; helps reduce overfitting (like bagging inside boosting).
# - Typical range: 0.5-1.0 (lower => stronger regularization).

# colsample
# - Fraction of features (columns) used when building a tree.
# - Prevents over-reliance on a few dominant features.
# - Improves generalization and diversity across trees (similar to random-forest feature sampling).

# learning_rate (eta)
# - Controls how much each tree contributes to the final prediction.
# - Lower value => slower learning but usually better generalization.
# - Smaller learning_rate often needs more trees (n_estimators).

# max_depth
# - Maximum depth of each decision tree.
# - Higher depth captures more complex patterns but increases overfitting risk.
# - Lower depth is simpler and acts as regularization.

**Why this works.** boosting fits residuals; learning rate shrinks each step; too many estimators can overfit.